# Notebook 14 – Computer Vision Data Pipeline



In [1]:
import time
import numpy as np
import torch
from torch import nn
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

PyTorch: 2.14.0+cpu | device: cpu


## Folder structure and leakage
```text
dataset/train/class_a/*.jpg
dataset/train/class_b/*.jpg
dataset/val/class_a/*.jpg
dataset/val/class_b/*.jpg
dataset/test/class_a/*.jpg
dataset/test/class_b/*.jpg
```
Keep splits disjoint. Group samples from the same patient, video, or object before splitting to prevent leakage. Validation/test transforms are deterministic; augmentation is train-only.

In [2]:
from pathlib import Path
from PIL import Image,UnidentifiedImageError
from collections import Counter
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
ROOT=Path('dataset'); suffixes={'.jpg','.jpeg','.png','.bmp','.webp'}
def validate_images(root):
 paths=[p for p in root.rglob('*') if p.suffix.lower() in suffixes]; good=[]; bad=[]
 for p in paths:
  try:
   with Image.open(p) as im: im.verify()
   good.append(p)
  except (UnidentifiedImageError,OSError,ValueError): bad.append(p)
 return good,bad
if ROOT.exists():
 good,bad=validate_images(ROOT); print('valid:',len(good),'corrupt:',len(bad),'examples:',bad[:5])
else: print('Set ROOT to a dataset folder; expected train/, val/, test/ subfolders.')
train_dir=ROOT/'train'; val_dir=ROOT/'val'
if train_dir.exists() and val_dir.exists():
 train_tf=transforms.Compose([transforms.Resize((128,128)),transforms.RandomHorizontalFlip(),transforms.ToTensor()]); eval_tf=transforms.Compose([transforms.Resize((128,128)),transforms.ToTensor()])
 train_ds=datasets.ImageFolder(train_dir,transform=train_tf); val_ds=datasets.ImageFolder(val_dir,transform=eval_tf)
 assert train_ds.class_to_idx==val_ds.class_to_idx, 'class mapping mismatch'
 print('mapping:',train_ds.class_to_idx,'counts:',Counter(train_ds.targets))
 train_loader=DataLoader(train_ds,batch_size=32,shuffle=True,num_workers=0); val_loader=DataLoader(val_ds,batch_size=64,shuffle=False,num_workers=0)
 print('batches:',len(train_loader),len(val_loader))

Set ROOT to a dataset folder; expected train/, val/, test/ subfolders.


## Interpret the pipeline
Compute channel statistics on training images only; inspect class counts and image-size distributions; log corrupt-file exclusions. If classes are imbalanced, use class-aware metrics and a considered balancing strategy. Keep class mapping stable across splits. A reusable pipeline has explicit, tested behavior at each boundary.